# 🦺 PPE (Personal Protective Equipment) Detection Training
## Google Colab Notebook

This notebook demonstrates how to train a **YOLOv8** model for detecting PPE items such as:
- 👷 Hard Hats / Helmets
- 🦺 Safety Vests
- 🧤 Gloves
- 🥽 Safety Goggles/Glasses
- 😷 Masks

**Requirements:**
- Google Colab with GPU runtime (T4 or better recommended)
- Roboflow account for dataset access (free tier works)

---

## 1. Install Dependencies and Check GPU

In [ ]:
# Check GPU availability
import torch

print("=" * 50)
print("GPU INFORMATION")
print("=" * 50)

if torch.cuda.is_available():
    print(f"✅ GPU is available!")
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ No GPU detected. Please enable GPU in Runtime > Change runtime type > GPU")
    print("   Note: Training will be very slow on CPU!")

print("=" * 50)
print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# Install required packages
!pip install -q ultralytics roboflow opencv-python-headless matplotlib seaborn pillow

# Verify installation
import ultralytics
ultralytics.checks()

print("\n✅ All dependencies installed successfully!")

In [ ]:
# Import all necessary libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import glob
import random
import yaml
from IPython.display import display, Image as IPImage, clear_output
import warnings
warnings.filterwarnings('ignore')

# Ultralytics imports
from ultralytics import YOLO

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✅ All libraries imported successfully!")

## 2. Mount Google Drive and Setup Directories

Mount your Google Drive to save models and datasets persistently.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directories
PROJECT_NAME = "PPE_Detection"
BASE_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME

# Create directories
DATASET_DIR = BASE_DIR / "dataset"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

for dir_path in [DATASET_DIR, MODELS_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {dir_path}")

print(f"\n📁 Project Base Directory: {BASE_DIR}")

## 3. Download and Prepare PPE Dataset

We'll use Roboflow to download a PPE detection dataset. You can:
1. Use the public dataset below, OR
2. Upload your own dataset to Roboflow and use your API key

**Popular PPE Datasets on Roboflow:**
- Construction Site Safety
- PPE Detection Dataset
- Hard Hat Workers Dataset

In [ ]:
# Download PPE Dataset from Roboflow
# Replace with your own API key and dataset details if using custom data

from roboflow import Roboflow

# -----------------------------------------------------------------
# OPTION 1: Using a public PPE dataset (recommended for testing)
# -----------------------------------------------------------------
# Replace these with your Roboflow credentials
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # Get from https://roboflow.com/settings/api
WORKSPACE = "roboflow-universe-projects"
PROJECT = "construction-site-safety"
VERSION = 30

# Initialize Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8", location=str(DATASET_DIR))

print(f"\n✅ Dataset downloaded to: {DATASET_DIR}")

In [ ]:
# Explore the dataset structure
DATA_YAML = DATASET_DIR / "data.yaml"

# Read the data.yaml file
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print("=" * 50)
print("DATASET CONFIGURATION")
print("=" * 50)
print(f"\n📊 Classes ({data_config['nc']} total):")
for i, cls in enumerate(data_config['names']):
    print(f"   {i}: {cls}")

# Count images in each split
for split in ['train', 'valid', 'test']:
    split_path = DATASET_DIR / split / 'images'
    if split_path.exists():
        num_images = len(list(split_path.glob('*')))
        print(f"\n📁 {split.capitalize()}: {num_images} images")
    else:
        print(f"\n📁 {split.capitalize()}: Not found")

## 4. Data Visualization and Exploration

Let's visualize some training images with their annotations to understand the dataset better.

In [ ]:
# Visualize sample images with bounding boxes
def plot_images_with_boxes(image_dir, label_dir, class_names, num_images=6):
    """Display sample images with their bounding box annotations."""
    images = list(Path(image_dir).glob('*.jpg')) + list(Path(image_dir).glob('*.png'))
    sample_images = random.sample(images, min(num_images, len(images)))
    
    # Color palette for different classes
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Read corresponding label file
        label_path = Path(label_dir) / f"{img_path.stem}.txt"
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    cls_id = int(parts[0])
                    x_center, y_center, bbox_w, bbox_h = map(float, parts[1:5])
                    
                    # Convert YOLO format to pixel coordinates
                    x1 = int((x_center - bbox_w/2) * w)
                    y1 = int((y_center - bbox_h/2) * h)
                    x2 = int((x_center + bbox_w/2) * w)
                    y2 = int((y_center + bbox_h/2) * h)
                    
                    # Draw bounding box
                    color = tuple(int(c * 255) for c in colors[cls_id % len(colors)][:3])
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, class_names[cls_id], (x1, y1-5), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(img_path.name[:30])
    
    plt.suptitle('Sample Training Images with PPE Annotations', fontsize=14)
    plt.tight_layout()
    plt.show()

# Plot sample images
train_images = DATASET_DIR / 'train' / 'images'
train_labels = DATASET_DIR / 'train' / 'labels'
plot_images_with_boxes(train_images, train_labels, data_config['names'])

In [ ]:
# Analyze class distribution in the dataset
def analyze_class_distribution(label_dir, class_names):
    """Count instances of each class in the dataset."""
    class_counts = {name: 0 for name in class_names}
    
    for label_file in Path(label_dir).glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f.readlines():
                cls_id = int(line.strip().split()[0])
                class_counts[class_names[cls_id]] += 1
    
    return class_counts

# Get class distribution for training set
class_dist = analyze_class_distribution(train_labels, data_config['names'])

# Plot class distribution
plt.figure(figsize=(12, 5))
colors = plt.cm.viridis(np.linspace(0, 0.8, len(class_dist)))
bars = plt.bar(class_dist.keys(), class_dist.values(), color=colors)
plt.xlabel('PPE Class')
plt.ylabel('Number of Instances')
plt.title('Class Distribution in Training Dataset')
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar, count in zip(bars, class_dist.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
             str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Class Distribution Summary:")
for cls, count in class_dist.items():
    print(f"   {cls}: {count} instances")

## 5. Define Model Architecture (YOLOv8)

YOLOv8 comes in different sizes. Choose based on your needs:
- **YOLOv8n** (Nano): Fastest, lowest accuracy - good for edge devices
- **YOLOv8s** (Small): Good balance for most use cases
- **YOLOv8m** (Medium): Better accuracy, moderate speed
- **YOLOv8l** (Large): High accuracy, slower
- **YOLOv8x** (Extra Large): Highest accuracy, slowest

In [ ]:
# Initialize YOLOv8 model
# Choose model size: 'yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt', 'yolov8l.pt', 'yolov8x.pt'
MODEL_SIZE = 'yolov8s.pt'  # Small model - good balance of speed and accuracy

# Load pretrained model (transfer learning)
model = YOLO(MODEL_SIZE)

print(f"✅ Loaded {MODEL_SIZE} with pretrained COCO weights")
print(f"   Model architecture: YOLOv8")
print(f"   Pretrained classes: 80 (COCO)")
print(f"   Will be fine-tuned for: {data_config['nc']} PPE classes")

## 6. Configure Training Parameters

Set up hyperparameters for training. Adjust these based on your dataset size and available GPU memory.

In [ ]:
# Training Configuration
# =======================

# Basic Training Parameters
EPOCHS = 100           # Number of training epochs (start with 50-100)
BATCH_SIZE = 16        # Batch size (reduce if OOM error, increase if GPU allows)
IMG_SIZE = 640         # Image size (640 is YOLOv8 default)
PATIENCE = 20          # Early stopping patience

# Learning Rate
LR0 = 0.01            # Initial learning rate
LRF = 0.01            # Final learning rate (lr0 * lrf)

# Optimizer
OPTIMIZER = 'auto'     # Options: 'SGD', 'Adam', 'AdamW', 'auto'
WEIGHT_DECAY = 0.0005  # L2 regularization

# Data Augmentation (built into YOLOv8)
AUGMENT = True         # Enable augmentation
MOSAIC = 1.0          # Mosaic augmentation probability
MIXUP = 0.0           # Mixup augmentation probability
COPY_PASTE = 0.0      # Copy-paste augmentation probability
DEGREES = 0.0         # Rotation (+/- degrees)
TRANSLATE = 0.1       # Translation (+/- fraction)
SCALE = 0.5           # Scale (+/- gain)
FLIPLR = 0.5          # Horizontal flip probability
FLIPUD = 0.0          # Vertical flip probability

# Save settings
SAVE_PERIOD = 10      # Save checkpoint every N epochs
PROJECT_DIR = str(MODELS_DIR)
RUN_NAME = "ppe_yolov8s_v1"

print("=" * 50)
print("TRAINING CONFIGURATION")
print("=" * 50)
print(f"""
📊 Training Settings:
   Epochs:        {EPOCHS}
   Batch Size:    {BATCH_SIZE}
   Image Size:    {IMG_SIZE}x{IMG_SIZE}
   
🎯 Optimizer:
   Type:          {OPTIMIZER}
   Learning Rate: {LR0}
   Weight Decay:  {WEIGHT_DECAY}
   
🔄 Augmentation:
   Mosaic:        {MOSAIC}
   Flip LR:       {FLIPLR}
   Scale:         {SCALE}
   
💾 Output:
   Project:       {PROJECT_DIR}
   Run Name:      {RUN_NAME}
""")

## 7. Train the Model 🚀

Now let's train the model! This will take some time depending on your dataset size and GPU.

**Tips:**
- Monitor the training progress in the output
- Watch for overfitting (val loss increasing while train loss decreases)
- The model will automatically save the best weights

In [ ]:
# Start Training!
print("🚀 Starting training...")
print("=" * 50)

results = model.train(
    data=str(DATA_YAML),           # Dataset configuration
    epochs=EPOCHS,                  # Number of epochs
    batch=BATCH_SIZE,              # Batch size
    imgsz=IMG_SIZE,                # Image size
    patience=PATIENCE,              # Early stopping patience
    
    # Learning rate
    lr0=LR0,                       # Initial LR
    lrf=LRF,                       # Final LR ratio
    
    # Optimizer
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    
    # Augmentation
    augment=AUGMENT,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    
    # Saving
    project=PROJECT_DIR,
    name=RUN_NAME,
    save_period=SAVE_PERIOD,
    save=True,
    
    # Other settings
    device=0,                      # GPU device (0 for first GPU)
    workers=2,                     # Number of data loader workers
    pretrained=True,               # Use pretrained weights
    verbose=True,                  # Verbose output
    seed=42,                       # Random seed
    
    # Visualization
    plots=True,                    # Generate training plots
)

print("\n" + "=" * 50)
print("✅ Training Complete!")
print("=" * 50)

In [ ]:
# Display training results plots
TRAIN_DIR = Path(PROJECT_DIR) / RUN_NAME

# Show training curves
results_img = TRAIN_DIR / 'results.png'
if results_img.exists():
    plt.figure(figsize=(15, 10))
    img = plt.imread(str(results_img))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results - Loss and Metrics over Epochs')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Results plot not found. Training may still be in progress.")

## 8. Evaluate Model Performance

Let's evaluate the trained model on the validation set and analyze its performance.

In [ ]:
# Load the best model weights
BEST_MODEL_PATH = TRAIN_DIR / 'weights' / 'best.pt'
best_model = YOLO(str(BEST_MODEL_PATH))

print(f"✅ Loaded best model from: {BEST_MODEL_PATH}")

# Run validation
print("\n🔍 Running validation...")
val_results = best_model.val(data=str(DATA_YAML), verbose=True)

# Print detailed metrics
print("\n" + "=" * 50)
print("VALIDATION METRICS")
print("=" * 50)
print(f"""
📊 Overall Performance:
   mAP50:         {val_results.box.map50:.4f}
   mAP50-95:      {val_results.box.map:.4f}
   
🎯 Detection Metrics:
   Precision:     {val_results.box.mp:.4f}
   Recall:        {val_results.box.mr:.4f}
""")

In [ ]:
# Display confusion matrix
confusion_matrix_img = TRAIN_DIR / 'confusion_matrix.png'
confusion_matrix_norm_img = TRAIN_DIR / 'confusion_matrix_normalized.png'

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

if confusion_matrix_img.exists():
    img1 = plt.imread(str(confusion_matrix_img))
    axes[0].imshow(img1)
    axes[0].axis('off')
    axes[0].set_title('Confusion Matrix (Counts)', fontsize=12)

if confusion_matrix_norm_img.exists():
    img2 = plt.imread(str(confusion_matrix_norm_img))
    axes[1].imshow(img2)
    axes[1].axis('off')
    axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12)

plt.suptitle('PPE Detection - Confusion Matrix Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Run Inference on Test Images

Let's test our trained model on some sample images and visualize the predictions!

In [ ]:
# Run inference on test images
TEST_IMAGES_DIR = DATASET_DIR / 'test' / 'images'
if not TEST_IMAGES_DIR.exists():
    TEST_IMAGES_DIR = DATASET_DIR / 'valid' / 'images'

# Get sample test images
test_images = list(TEST_IMAGES_DIR.glob('*.jpg')) + list(TEST_IMAGES_DIR.glob('*.png'))
sample_images = random.sample(test_images, min(6, len(test_images)))

print(f"🔍 Running inference on {len(sample_images)} test images...")

# Run inference
results = best_model.predict(
    source=sample_images,
    conf=0.25,           # Confidence threshold
    iou=0.45,            # NMS IOU threshold
    save=True,           # Save results
    project=str(RESULTS_DIR),
    name='inference_results'
)

print("✅ Inference complete!")

In [ ]:
# Visualize inference results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, result in enumerate(results[:6]):
    # Get the image with annotations
    annotated_img = result.plot()  # Returns BGR image with annotations
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(annotated_img)
    axes[idx].axis('off')
    
    # Get detection info
    num_detections = len(result.boxes)
    axes[idx].set_title(f'{Path(result.path).name}\n{num_detections} detections', fontsize=10)

plt.suptitle('PPE Detection Results on Test Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print detection details
print("\n📋 Detection Summary:")
print("-" * 60)
for result in results[:6]:
    img_name = Path(result.path).name
    boxes = result.boxes
    print(f"\n📸 {img_name}:")
    if len(boxes) > 0:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = result.names[cls_id]
            print(f"   • {cls_name}: {conf:.2%} confidence")
    else:
        print("   No PPE detected")

## 10. Save and Export the Trained Model

Export your model in various formats for deployment:
- **PyTorch (.pt)**: Default format, best for Python inference
- **ONNX (.onnx)**: Cross-platform, good for various frameworks
- **TensorRT (.engine)**: Optimized for NVIDIA GPUs
- **TFLite**: For mobile/edge deployment
- **CoreML**: For Apple devices

In [ ]:
# Export model in various formats
print("📦 Exporting model in different formats...")
print("=" * 50)

# Export to ONNX (recommended for cross-platform deployment)
onnx_path = best_model.export(format='onnx', dynamic=True, simplify=True)
print(f"✅ ONNX exported: {onnx_path}")

# Export to TorchScript (for PyTorch deployment without Python)
torchscript_path = best_model.export(format='torchscript')
print(f"✅ TorchScript exported: {torchscript_path}")

# Optional: Export to other formats (uncomment as needed)
# tflite_path = best_model.export(format='tflite')  # TensorFlow Lite
# engine_path = best_model.export(format='engine')   # TensorRT (requires NVIDIA GPU)
# coreml_path = best_model.export(format='coreml')   # CoreML (for Apple devices)

print("\n" + "=" * 50)
print("✅ Model exports complete!")

In [ ]:
# Copy best model to Google Drive for safekeeping
import shutil

# Copy the best.pt model to the models directory
final_model_path = MODELS_DIR / 'ppe_detector_best.pt'
shutil.copy(BEST_MODEL_PATH, final_model_path)

print("=" * 50)
print("📁 MODEL SAVED TO GOOGLE DRIVE")
print("=" * 50)
print(f"""
✅ Best Model:    {final_model_path}
✅ Training Dir:  {TRAIN_DIR}
✅ Results Dir:   {RESULTS_DIR}

📌 To use this model later:
   from ultralytics import YOLO
   model = YOLO('{final_model_path}')
   results = model.predict('your_image.jpg')
""")

## 11. Bonus: Single Image Detection Function

A reusable function to detect PPE in any image.

In [ ]:
def detect_ppe(image_path, model, conf_threshold=0.25, show=True):
    """
    Detect PPE items in an image.
    
    Args:
        image_path: Path to the image
        model: Loaded YOLO model
        conf_threshold: Confidence threshold for detections
        show: Whether to display the result
        
    Returns:
        dict: Detection results with classes, confidences, and bounding boxes
    """
    # Run inference
    results = model.predict(image_path, conf=conf_threshold, verbose=False)[0]
    
    # Extract detection information
    detections = []
    for box in results.boxes:
        det = {
            'class': results.names[int(box.cls[0])],
            'confidence': float(box.conf[0]),
            'bbox': box.xyxy[0].tolist()  # [x1, y1, x2, y2]
        }
        detections.append(det)
    
    # Display result
    if show:
        plt.figure(figsize=(12, 8))
        annotated_img = results.plot()
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        plt.imshow(annotated_img)
        plt.axis('off')
        plt.title(f'PPE Detection Results - {len(detections)} items found')
        plt.show()
    
    # Print summary
    print(f"\n🔍 Found {len(detections)} PPE items:")
    for det in detections:
        print(f"   • {det['class']}: {det['confidence']:.1%}")
    
    return detections

# Example usage - test with a random image
random_test_img = random.choice(test_images)
print(f"Testing with: {random_test_img.name}")
detections = detect_ppe(random_test_img, best_model)

---

## 🎉 Summary

Congratulations! You've successfully trained a PPE detection model using YOLOv8. Here's what we covered:

1. ✅ **Environment Setup** - GPU check, library installation
2. ✅ **Data Preparation** - Downloaded and explored PPE dataset from Roboflow
3. ✅ **Data Visualization** - Analyzed class distribution and sample images
4. ✅ **Model Training** - Fine-tuned YOLOv8 with data augmentation
5. ✅ **Evaluation** - Analyzed mAP, precision, recall, and confusion matrix
6. ✅ **Inference** - Tested on new images
7. ✅ **Export** - Saved model in multiple formats for deployment

### Next Steps:
- 🔄 **Improve Performance**: Train longer, use larger model (YOLOv8m/l), add more data
- 📱 **Deploy**: Use ONNX/TFLite exports for web/mobile apps
- 🎥 **Real-time**: Apply to video streams for live PPE monitoring
- 🔧 **Fine-tune**: Collect domain-specific data for your use case

### Resources:
- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com/)
- [Roboflow Universe](https://universe.roboflow.com/) - More datasets
- [Roboflow Blog](https://blog.roboflow.com/) - Tutorials and tips